# Universal Differential Equations

A **Universal Differential Equation (UDE)** combines a mechanistic ODE with a neural network to learn unknown or partially-known dynamics from data. The ODE encodes prior biological knowledge; the neural network (called the NDE component) learns residual dynamics the ODE cannot explain on its own.

This notebook demonstrates UDE training on a Lotka-Volterra predator-prey system with an unknown external forcing term.

In [ ]:
import diffrax
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import pandas as pd

from example_models import get_lotka_volterra
from mxlpy import plot
from mxlpy.jax.models import Node, Ode, Ude
from mxlpy.jax.train import IntegrationSettings, train

In [ ]:
lv = Ode.from_mxlpy(get_lotka_volterra())
ts = jnp.linspace(0, 50, 100)
y0 = jnp.array([10.0, 10.0])
ys = lv.integrate(ts, y0)

fig, ax = plt.subplots()
ax.plot(ts, ys)
ax.legend(["prey", "pred"])
plt.show()

## Baseline: mechanistic ODE

The Lotka-Volterra system describes predator-prey dynamics with four parameters (`a`, `b`, `c`, `d`). This serves as the mechanistic prior — the part of the system we already understand and can encode explicitly.

In [ ]:
ys_forced = (ys.T + 0.14 * jnp.sin(ts)).T

fig, ax = plt.subplots()
ax.plot(ts, ys_forced)
ax.legend(["prey", "pred"])
plt.show()

## Training the UDE

We construct a `Ude` by combining the mechanistic `Ode` with a small neural network (`Node`). The `op="+"` means the NDE output is **added** to the ODE right-hand side at each time step.

Training uses `train`, which freezes the ODE parameters and only optimises the neural network weights. A curriculum schedule gradually increases the proportion of the trajectory used for loss computation, which helps avoid poor local minima early in training.

In [ ]:
ude, losses = train(
    Ude(
        ode=lv,
        nn=Node(
            n_obs=2,
            width=8,
            depth=3,
            key=jax.random.PRNGKey(42),
            out_scale=jnp.array([0.1]),
        ),
        op="+",
    ),
    ts=ts,
    ys=ys_forced,
    training_steps=[
        (500, 0.2),
        (500, 0.5),
        (10_000, 1.0),  # in practice longer, shortened for docs
    ],
    avg_every=50,
    integration_settings=IntegrationSettings(
        max_steps=8192,
        method=diffrax.Tsit5,
    ),
    optim=optax.adabelief(learning_rate=2e-4),
)

ys_pred = ude.integrate(ts, y0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
# ax1
ax1.plot(ts, ys_forced)
plot.reset_prop_cycle(ax=ax1)
ax1.plot(ts, ys_pred, ls="dashed")
ax1.legend(["prey", "pred"])

# ax2
pd.Series(losses, dtype=float)[12:].plot(ax=ax2)
plt.show()

## Separate the contribution of each model part

After training we can evaluate the ODE and NDE components independently along the predicted trajectory. `jax.vmap` maps each component over all time points in one call. This decomposition reveals what the neural network actually learned — ideally it should recover the unknown forcing signal.

In [ ]:
rhs_ode = jax.vmap(ude.ode, in_axes=(0, 0, None))(ts, ys_pred, jnp.array([]))
rhs_nde = jax.vmap(ude.nn, in_axes=(0, 0, None))(ts, ys_pred, jnp.array([]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
ax1.set(title="ODE")
ax2.set(title="NDE")
ax1.plot(ts, rhs_ode)
ax2.plot(ts, rhs_nde)

for ax in (ax1, ax2):
    ax.legend(["dpreydt", "dpreddt"])

The **ODE** panel shows the mechanistic dynamics encoded in the prior model. The **NDE** panel shows the learned correction — the neural network's contribution that accounts for dynamics the ODE could not explain. A well-trained UDE produces an NDE signal that approximates the unknown forcing term.